<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_2/lessons/lesson_25_pytest_testing/note_lesson_25_pytest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Урок 25 — Тестування з pytest (+ валідація AI-коду)

У п'ятницю хтось «підчистив» тариф таксі «Смачно + Таксі» — і мінімальна ціна 80 грн перестала діяти. Руками перевірили одну поїздку на 10 км, і вона була правильною. Сьогодні пишемо перевірки, які повторюються після кожної зміни:

1. Перший тест, AAA, читання падіння.
2. Межі, `pytest.raises`, `parametrize`.
3. Fixtures і фабрики.
4. `Mock` для SMS-шлюзу.
5. Перевірка AI-коду тестами на інваріанти.

**Як працює ноутбук:** код і тести записуються у файли папки `lesson25_lab/` (`%%writefile`), а функція `run_pytest` запускає pytest у цій папці. Виконуй клітинки **зверху вниз**. Теорія й архітектура — у книзі: [Урок 25. Тестування з pytest](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m2/lesson_25/).

## 🔁 Пригадай (без підглядання)

1. Що робить `assert`, якщо умова хибна?
2. Як у командному проєкті уроку 15 тести однієї задачі не залежали від інших?
3. Чому стадії конвеєра з уроку 24 легко перевіряти окремо?

<details>
<summary>Відповіді</summary>

1. Кидає `AssertionError`.
2. Замість функцій інших учасників підставлялися готові заглушки.
3. Кожна стадія приймає будь-яке ітерабельне — досить списку з кількох рядків.

</details>

## ⚙️ Підготовка

Якщо pytest не встановлено, виконай `!pip install pytest` (у Colab він уже є).

In [ ]:
import os
import subprocess
import sys

LAB = "lesson25_lab"
os.makedirs(LAB, exist_ok=True)


def run_pytest(*args):
    """Запускає pytest у папці LAB і повертає код виходу: 0 — усі тести пройшли."""
    result = subprocess.run(
        [sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", *args],
        cwd=LAB, capture_output=True, text=True,
    )
    print((result.stdout or result.stderr)[-1500:])
    return result.returncode

## 1. Перший тест

Код тарифу — у файл `pricing.py`:

In [ ]:
%%writefile lesson25_lab/pricing.py
"""Тарифи й знижки."""

from dataclasses import dataclass


@dataclass(frozen=True)
class Tariff:
    base: int       # подача, грн
    per_km: int     # грн за кілометр
    min_fare: int   # мінімальна вартість поїздки, грн


def fare(km, tariff):
    """Вартість поїздки на km кілометрів, не менша за мінімальну."""
    if km < 0:
        raise ValueError(f"відстань не може бути від'ємною: {km}")
    return max(tariff.min_fare, round(tariff.base + tariff.per_km * km))


def apply_promo(amount, percent):
    """Сума після знижки percent % (від 0 до 50), округлена до гривні."""
    if not 0 <= percent <= 50:
        raise ValueError(f"знижка має бути від 0 до 50 %, а маємо {percent}")
    return round(amount * (100 - percent) / 100)

In [ ]:
%%writefile lesson25_lab/test_first.py
from pricing import Tariff, fare


def test_long_trip_is_base_plus_distance():
    # Arrange
    tariff = Tariff(base=40, per_km=12, min_fare=80)
    # Act
    price = fare(10, tariff)
    # Assert
    assert price == 160


def test_short_trip_costs_min_fare():
    assert fare(0.5, Tariff(base=40, per_km=12, min_fare=80)) == 80

In [ ]:
assert run_pytest("test_first.py") == 0

### Читаємо падіння

**Прогноз:** нижче — «п'ятнична» версія тарифу без `max`. Який із двох тестів упаде і що покаже pytest у рядку `assert`?

In [ ]:
%%writefile lesson25_lab/test_broken_demo.py
from pricing import Tariff


def fare_after_friday(km, tariff):
    return round(tariff.base + tariff.per_km * km)


def test_long_trip():
    assert fare_after_friday(10, Tariff(40, 12, 80)) == 160


def test_short_trip():
    assert fare_after_friday(0.5, Tariff(40, 12, 80)) == 80

In [ ]:
assert run_pytest("test_broken_demo.py") != 0, "тут падіння — це очікувано"

<details>
<summary>Відповідь</summary>

Падає лише `test_short_trip`: `assert 46 == 80`, а рядок `where` показує, звідки взялося 46. Поїздка на 10 км проходить — саме її й перевірили руками в п'ятницю.

</details>

## 🛠 Вправа 1. Тести для промокоду

`apply_promo(amount, percent)` з `pricing.py`: знижка від 0 до 50 %, результат округлено до гривні; інше — `ValueError`. Напиши у `test_promo.py`:

- параметризований тест щонайменше на 3 звичайні випадки (серед них — 0 %);
- тест на межу 50 %;
- `pytest.raises(ValueError)` для −5 і 51.

In [ ]:
%%writefile lesson25_lab/test_promo.py
import pytest

from pricing import apply_promo

# YOUR CODE HERE
# BEGIN SOLUTION
@pytest.mark.parametrize("amount, percent, expected", [(340, 10, 306), (340, 0, 340), (95, 15, 81)])
def test_apply_promo(amount, percent, expected):
    assert apply_promo(amount, percent) == expected


def test_max_promo_is_half():
    assert apply_promo(200, 50) == 100


@pytest.mark.parametrize("percent", [-5, 51])
def test_bad_percent(percent):
    with pytest.raises(ValueError):
        apply_promo(340, percent)
# END SOLUTION

In [ ]:
assert run_pytest("test_promo.py") == 0
text = open(f"{LAB}/test_promo.py", encoding="utf-8").read()
assert "parametrize" in text and "pytest.raises" in text, "потрібні parametrize і pytest.raises"
print("✅ Вправа 1 пройдена")

## 🛠 Вправа 2. Знайди баг тестом

Правило: доставка безкоштовна для замовлень **від 500 грн** (включно), інакше — 60 грн. Напиши `test_fee.py` для `delivery_fee(total)` з модуля `fee`. Перевірка нижче запустить твої тести **двічі**: на версії з помилкою (вони мусять упасти) і на правильній (мусять пройти).

In [ ]:
%%writefile lesson25_lab/test_fee.py
import pytest

from fee import delivery_fee

# YOUR CODE HERE
# BEGIN SOLUTION
@pytest.mark.parametrize("total, expected", [(0, 60), (499, 60), (500, 0), (501, 0), (1200, 0)])
def test_delivery_fee(total, expected):
    assert delivery_fee(total) == expected
# END SOLUTION

In [ ]:
BUGGY = "def delivery_fee(total):\n    return 0 if total > 500 else 60\n"
CORRECT = "def delivery_fee(total):\n    return 0 if total >= 500 else 60\n"

with open(f"{LAB}/fee.py", "w", encoding="utf-8") as file:
    file.write(BUGGY)
assert run_pytest("test_fee.py") == 1, "твої тести не помітили помилку на межі 500 (або тестів ще немає)"

with open(f"{LAB}/fee.py", "w", encoding="utf-8") as file:
    file.write(CORRECT)
assert run_pytest("test_fee.py") == 0, "на правильному коді тести мають пройти"
print("✅ Вправа 2 пройдена")

## 🛠 Вправа 3. Fixtures для конвеєра подій

Конвеєр з уроку 24 — у файл `events.py`. У `test_events_lab.py` напиши fixture-фабрику `make_event(order=1, kind="picked", time="18:00", courier="D-1")`, що повертає рядок події, і тести:

- `durations` рахує час між `picked` і `delivered` одного замовлення;
- `delivered` без `picked` пропускається;
- `parse` відкладає битий рядок у `rejected`.

In [ ]:
%%writefile lesson25_lab/events.py
"""Конвеєр подій кур'єрів з уроку 24: кожна стадія — «ітерабельне → ітерабельне»."""

from itertools import dropwhile


def minutes(hhmm):
    hours, mins = hhmm.split(":")
    return int(hours) * 60 + int(mins)


def parse(lines, rejected):
    for line in lines:
        parts = line.split()
        if len(parts) != 4 or not parts[2].isdigit():
            rejected.append(line)
            continue
        time, courier, order, kind = parts
        yield {"time": minutes(time), "courier": courier, "order": int(order), "kind": kind}


def in_shift(events, start):
    return dropwhile(lambda event: event["time"] < start, events)


def durations(events):
    picked = {}
    for event in events:
        if event["kind"] == "picked":
            picked[event["order"]] = event["time"]
        elif event["kind"] == "delivered" and event["order"] in picked:
            yield event["courier"], event["time"] - picked.pop(event["order"])


def report(pairs):
    by_courier = {}
    for courier, spent in pairs:
        by_courier.setdefault(courier, []).append(spent)
    return {courier: round(sum(spent) / len(spent), 1) for courier, spent in sorted(by_courier.items())}


def read_log(path):
    with open(path, encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if line:
                yield line

In [ ]:
%%writefile lesson25_lab/test_events_lab.py
import pytest

from events import durations, parse

# YOUR CODE HERE
# BEGIN SOLUTION
@pytest.fixture
def make_event():
    def factory(order=1, kind="picked", time="18:00", courier="D-1"):
        return f"{time} {courier} {order} {kind}"
    return factory


def test_duration_of_one_order(make_event):
    lines = [make_event(kind="picked", time="18:00"), make_event(kind="delivered", time="18:25")]
    assert list(durations(parse(lines, []))) == [("D-1", 25)]


def test_delivered_without_picked_is_skipped(make_event):
    assert list(durations(parse([make_event(kind="delivered")], []))) == []


def test_broken_line_is_rejected(make_event):
    rejected = []
    events = list(parse([make_event(), "битий рядок"], rejected))
    assert len(events) == 1 and rejected == ["битий рядок"]
# END SOLUTION

In [ ]:
assert run_pytest("test_events_lab.py") == 0
assert "def make_event" in open(f"{LAB}/test_events_lab.py", encoding="utf-8").read()
print("✅ Вправа 3 пройдена")

## 🛠 Вправа 4. Mock SMS-шлюзу

`notify_client(order_id, phone, minutes, gateway)` надсилає SMS через `gateway.send`. Напиши `test_notify_lab.py` з `unittest.mock.Mock`:

- перевір текст і номер через `assert_called_once_with`;
- через `side_effect = ConnectionError(...)` перевір, що функція повертає `False` і не падає.

In [ ]:
%%writefile lesson25_lab/notify.py
"""Сповіщення клієнта. Шлюз SMS передається параметром — його легко підмінити в тесті."""


class SmsGateway:
    """Справжній шлюз: ходить у мережу й коштує гроші за кожне повідомлення."""

    def send(self, phone, text):
        raise ConnectionError("справжня мережа недоступна в навчальному проєкті")


def notify_client(order_id, phone, minutes, gateway):
    """Надсилає клієнту час доставки. Повертає True, якщо SMS пішло."""
    text = f"Замовлення №{order_id} буде за {minutes} хв"
    try:
        gateway.send(phone, text)
    except ConnectionError:
        return False
    return True

In [ ]:
%%writefile lesson25_lab/test_notify_lab.py
from unittest.mock import Mock

from notify import notify_client

# YOUR CODE HERE
# BEGIN SOLUTION
def test_sms_is_sent():
    gateway = Mock()
    assert notify_client(7, "+380670000000", 40, gateway) is True
    gateway.send.assert_called_once_with("+380670000000", "Замовлення №7 буде за 40 хв")


def test_gateway_down():
    gateway = Mock()
    gateway.send.side_effect = ConnectionError("шлюз не відповідає")
    assert notify_client(7, "+380670000000", 40, gateway) is False
# END SOLUTION

In [ ]:
assert run_pytest("test_notify_lab.py") == 0
text = open(f"{LAB}/test_notify_lab.py", encoding="utf-8").read()
assert "side_effect" in text and "assert_called_once_with" in text
print("✅ Вправа 4 пройдена")

## 🛠 Вправа 5. Перевір AI-код

AI написав `split_bill(total, people)` — «розділи рахунок порівну». Напиши у `test_split.py` тести на **властивості** для багатьох комбінацій `total` і `people`:

1. частин рівно `people`;
2. сума частин дорівнює `total`;
3. частини відрізняються щонайбільше на 1;
4. `people=0` → `ValueError`.

Перевірка запустить твої тести на AI-версії (мусять упасти) і на правильній (мусять пройти).

In [ ]:
%%writefile lesson25_lab/test_split.py
import pytest

from split import split_bill

# YOUR CODE HERE
# BEGIN SOLUTION
@pytest.mark.parametrize("people", range(1, 8))
@pytest.mark.parametrize("total", range(0, 301, 7))
def test_split_properties(total, people):
    shares = split_bill(total, people)
    assert len(shares) == people
    assert sum(shares) == total
    assert max(shares) - min(shares) <= 1


def test_zero_people():
    with pytest.raises(ValueError):
        split_bill(100, 0)
# END SOLUTION

In [ ]:
AI_VERSION = """
def split_bill(total, people):
    share = round(total / people)
    return [share] * people
"""
CORRECT = """
def split_bill(total, people):
    if people < 1:
        raise ValueError("людей має бути щонайменше 1")
    share, rest = divmod(total, people)
    return [share + 1] * rest + [share] * (people - rest)
"""

with open(f"{LAB}/split.py", "w", encoding="utf-8") as file:
    file.write(AI_VERSION)
assert run_pytest("test_split.py") == 1, "твої тести пропустили помилки AI-версії (або тестів ще немає)"

with open(f"{LAB}/split.py", "w", encoding="utf-8") as file:
    file.write(CORRECT)
assert run_pytest("test_split.py") == 0, "на правильній версії тести мають пройти"
print("✅ Вправа 5 пройдена")

## ✅ Самоперевірка

1. Чому п'ятничний баг не зловив тест на 10 км?
2. Як pytest дізнається, яку fixture передати в тест?
3. Навіщо в перевірці вправи 2 тести запускаються на зламаному коді?
4. Що мокати, а що ні?
5. Чому тест, що повторює формулу реалізації, небезпечний?

<details>
<summary>Відповіді</summary>

1. На 10 км мінімум не діє — баг проявляється лише нижче межі.
2. За іменем параметра тестової функції.
3. Тест, який проходить на будь-якому коді, нічого не перевіряє.
4. Межі системи: мережу, SMS, платежі, час. Власну логіку — ні.
5. Помилка у формулі потрапляє і в очікуване значення — тест проходить на неправильному коді.

</details>

### Шпаргалка

```python
def test_name():                         # файл test_*.py, функція test_*
    assert fare(10, day) == 160          # AAA: Arrange, Act, Assert

with pytest.raises(ValueError, match="від'ємною"):
    fare(-1, day)                        # лише один рядок усередині
assert 351.666 == pytest.approx(351.67, abs=0.01)

@pytest.mark.parametrize("km, expected", [(0, 80), (10, 160)], ids=["zero", "long"])
@pytest.fixture                          # у tests/conftest.py — для всіх файлів
def day_tariff(): return Tariff(40, 12, 80)

gateway = Mock(); gateway.send.side_effect = ConnectionError()
gateway.send.assert_called_once_with(phone, text)
with patch("delivery.services.send_sms"): ...   # там, де ім'я використовується
```

`python -m pytest -v -k fare` · `-s` (показати print) · `--cov=delivery --cov-report=term-missing`

## Далі

- **Проєкт `delivery_tests/`** у папці уроку — 32 тести; спробуй зламати код за `README.md`.
- **Практикум `basics/`** — 5 файлів від першого `assert` до `parametrize`.
- **Урок 26 — П5. Динамічне програмування**: тестами перевіримо, що швидке рішення збігається з повільним.